# Sistemas de recomendación basados en contenido (Content Based Systems)



**Librería Pandas**

Pandas es una librería de código abierto creada para los desarrolladores en Python, la cual es utilizada en su mayoría en las áreas de Aprendizaje automático y Ciencia de datos. El objetivo que se estableció al crear esta librería fue de dotar de las herramientas necesarias a los analistas de datos, es decir, de las principales funcionalidades que permiten la manipulación y el tratamiento de los datos.

**Librería NumPy**

NumPy es una librería de Python desarrollada en mayor proporción en el lenguaje de programación C, que permite el uso de funciones matemáticas y numéricas en cortos tiempos de ejecución. También provee de estructuras de datos, que permiten realizar cálculos de matrices de forma eficiente con grandes cantidades de datos.

La función "dot" de NumPy permite el cálculo del producto punto entre dos matrices como parámetro de entrada. La función "linalg.norm" de NumPy permite devolver una de los ocho normas matriciales diferentes, o una de un número infinito de normas vectoriales, dependiendo del valor del parámetro ord.

In [1]:
# Importación de librerías necesarias
import pandas as pd
import numpy as np
from numpy import dot
from numpy.linalg import norm 

In [2]:
# Declaración del dataset en una constante
PATH = 'data.csv'

# Importar Datos

In [3]:
# Creación del marco de datos y verificación de sus dimensiones
df = pd.read_csv(PATH)
df.shape

(100000, 10)

In [4]:
df.head()

,book_id,author_id,book_genre,reader_id,num_pages,book_rating,publisher_id,publish_year,book_price,text_lang
0,655,52,4,11482,300,4,8,2012,94,7
1,2713,90,3,6479,469,1,8,2012,33,5
2,409,17,2,25472,435,1,12,2001,196,4
3,1150,234,10,23950,529,2,23,2019,79,2
4,2424,390,5,13046,395,2,20,2010,200,4


# Recomendación de Libros

In [5]:
def normalizar(data):
    '''
    Esta función normalizará los datos de entrada para que estén entre 0 y 1
    
    parámetros:
        data (Lista) : La lista de valores que desea normalizar
    
    retorna:
        Los datos de entrada normalizados entre 0 y 1
    '''
    min_val = min(data)
    if min_val < 0:
        data = [x + abs(min_val) for x in data]
    max_val = max(data)
    return [x/max_val for x in data]

In [6]:
# Normalizar las columnas num_pages, book_rating, book_price
df['num_pages_norm'] = normalizar(df['num_pages'].values)
df['book_rating_norm'] = normalizar(df['book_rating'].values)
df['book_price_norm'] = normalizar(df['book_price'].values)

In [7]:
def ohe(df, enc_col):
    '''
    Esta función realizará el método one hot encode a la columna especificada y la añadirá
    en el marco de datos de entrada
    
    parámetros:
        df (DataFrame) : El marco de datos al que desea añadir los resultados
        enc_col (String) : La columna que desea que utilice el método OHE
    
    retorna:
        Las columnas obtenidas por el método OHE añadidas al marco de datos de entrada
    '''
    
    ohe_df = pd.get_dummies(df[enc_col])
    ohe_df.reset_index(drop = True, inplace = True)
    return pd.concat([df, ohe_df], axis = 1)

In [8]:
# OHE en publish_year, book_genre y text_lang
df = ohe(df = df, enc_col = 'publish_year')
df = ohe(df = df, enc_col = 'book_genre')
df = ohe(df = df, enc_col = 'text_lang')

In [9]:
# Eliminar las columnas redundantes
cols = ['publish_year', 'book_genre', 'num_pages', 'book_rating', 'book_price', 'text_lang']
df.drop(columns = cols, inplace = True)
df.set_index('book_id', inplace = True)

In [10]:
class CBRecommend():
    def __init__(self, df):
        self.df = df
        
    def cosine_sim(self, v1,v2):
        '''
        Esta función calculará la similitud del coseno entre dos vectores
        '''
        return dot(v1,v2)/(norm(v1)*norm(v2))
    
    def recommend(self, book_id, n_rec):
        """
        df (dataframe): El marco de datos
        book_id (string): Representación del nombre del libro
        n_rec (int): cantidad de recomendaciones que quiere el usuario
        """
        
        # calcula la similitud del vector book_id de entrada con todos los demás vectores
        inputVec = self.df.loc[book_id].values
        self.df['sim']= self.df.apply(lambda x: self.cosine_sim(inputVec,x.values), axis=1)
        
        # devuelve los n mejores libros especificados por el usuario
        return self.df.nlargest(columns='sim',n=n_rec)

In [11]:
# Se ejecuta en una muestra de 1000 como ejemplo
t = df.sample(1000).copy()
cbr = CBRecommend(df = t)

In [12]:

cbr.df.head()

,author_id,reader_id,publisher_id,num_pages_norm,book_rating_norm,book_price_norm,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,1,2,3,4,5,6,7,8,9,10,1,2,3,4,5,6,7
book_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1209,179,18819,36,0.718571,0.1,0.310,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
367,23,8710,20,0.207143,0.4,0.650,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
2785,86,21751,34,0.711429,0.8,0.225,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1945,278,3663,2,0.894286,0.3,0.110,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2504,166,9754,38,0.157143,0.4,0.240,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0


In [13]:
# Muestra una cantidad de libros recomendados según la variable n_rec 
cbr.recommend(book_id = t.index[0], n_rec = 5)

,author_id,reader_id,publisher_id,num_pages_norm,book_rating_norm,book_price_norm,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,1,2,3,4,5,6,7,8,9,10,1,2,3,4,5,6,7,sim
book_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1209,179,18819,36,0.718571,0.1,0.310,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1.0
866,238,24469,45,0.702857,0.5,0.375,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1.0
975,244,26301,46,0.984286,0.8,0.065,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1.0
1464,265,28470,48,0.707143,0.8,0.655,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1.0
723,195,20160,45,0.237143,0.2,0.200,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1.0
